Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.carriers"
silver_table = f"{catalog}.{silver_schema}.carriers"

Read bronze Delta table into a DataFrame

In [0]:
bronze_carriers_df=(
    spark.read
    .format("delta")
    .table(bronze_table)
)

Drop null records

In [0]:
bronze_carriers_df=bronze_carriers_df.dropna()

Drop duplicated records

In [0]:
bronze_carriers_df=bronze_carriers_df.dropDuplicates()

Select and rename columns for clarity and unification

In [0]:
from pyspark.sql import functions as F

silver_carriers_df=(
    bronze_carriers_df
    .select(
        F.col("code").alias("carrier_key"),
        F.col("description").alias("airline_name"),
        "batch_id"
    )
)

Add created and updated timestamp columns

In [0]:
silver_carriers_df=(
    silver_carriers_df
    .withColumns({
        "created_timestamp":F.current_timestamp(),
        "updated_timestamp":F.current_timestamp()
    })
)

Write DataFrame to silver Delta table 

Subsequent runs merge new batch into existing table, only updating records from a newer or equal batch to avoid reprocessing

In [0]:
from pyspark.sql import Window

window = Window.partitionBy("carrier_key").orderBy(
    F.col("batch_id").desc()
)

silver_carriers_df= (
    silver_carriers_df
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)


if not spark.catalog.tableExists(silver_table):
    silver_carriers_df_write=(
        silver_carriers_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

else:
    from delta.tables import DeltaTable

    delta_table=DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            silver_carriers_df.alias("s"),
            "t.carrier_key = s.carrier_key"
        )
        .whenMatchedUpdate(
            condition="s.batch_id >= t.batch_id",
            set={
                "carrier_key": "s.carrier_key",
                "airline_name": "s.airline_name",
                "batch_id": "s.batch_id",
                "updated_timestamp": "s.updated_timestamp"
            }

        )
        .whenNotMatchedInsertAll()
        .execute()
    )